# 310 — Cell-Line Transcriptomic Program Discovery

## Objective

Identify candidate transcriptomic programs in the frozen cell-line modeling cohort using phenotype-independent latent-structure discovery, and determine whether the resulting representations are supported across complementary decomposition methods.

The primary discovery method is Independent Component Analysis (ICA). Non-negative Matrix Factorization (NMF) is used as a prespecified methodological contrast rather than as a competing procedure for selecting whichever representation shows the strongest pharmacological association.

Specifically, this notebook will:

- construct transcriptomic latent representations from the frozen expression matrix;
- evaluate numerical stability of ICA across deterministic multiple initializations;
- define a stable primary ICA solution without using the resistance-like phenotype;
- derive an independent NMF representation from the same frozen model and gene universe;
- compare ICA and NMF programs using model scores and gene-level structure;
- identify cross-method structural support without matching components by numerical index;
- associate frozen transcriptomic programs with the selected resistance-like phenotype only after program discovery is complete.

## Scope

This notebook is part of the Cell-Line Discovery Layer.

It does not reconstruct the modeling cohort, repeat transcriptomic quality control, redefine the pharmacological phenotype, select decomposition parameters based on pharmacological associations, perform tumor–cell-line matching, or establish cross-cancer consensus programs.

Lineage, alternative phenotype representations, drug-response coverage, additional cell-line covariates, and resampling-based program robustness will be evaluated in notebook 311.

Tumor–cell-line comparison and consensus construction remain part of Phase 4.

## Conceptual distinction

Program discovery answers:

> Which recurrent transcriptomic structures can be identified in the frozen cell-line expression space without using the pharmacological phenotype to construct them?

Phenotype association answers:

> Which of those independently defined transcriptomic structures are computationally associated with the frozen resistance-like phenotype?

Cross-method comparison answers:

> Does a transcriptomic structure identified by ICA have related support under NMF, a decomposition with different mathematical constraints?

These questions are kept separate to reduce phenotype-driven feature selection and post hoc method selection.

## Methodological strategy

ICA is the primary program-discovery framework because it can represent signed, partially independent transcriptomic axes with gene-level loadings.

NMF is used as an orthogonal structural contrast because it represents the same transcriptomic system through non-negative, additive factors.

The two methods are not expected to produce one-to-one equivalent components. Cross-method support will therefore be evaluated using score concordance and gene-level structure rather than component numbering.

The phenotype will not be used to choose ICA seeds, select stable components, determine ICA–NMF correspondence, or tune decomposition parameters.

The exploratory audit showed that ICA representations should not be assumed to be portable solely from component indices when the feature universe, training cohort, or factorization changes. Program identity will therefore be based on quantitative representation-level evidence rather than component labels alone. :contentReference[oaicite:0]{index=0} :contentReference[oaicite:1]{index=1}

## Expected outputs

This notebook will generate:

```text
data/processed/cellline_programs/310_ica_program_scores.parquet
data/processed/cellline_programs/310_ica_program_loadings.parquet
data/processed/cellline_programs/310_nmf_program_scores.parquet
data/processed/cellline_programs/310_nmf_program_loadings.parquet
data/processed/cellline_programs/310_cross_method_program_matching.csv
data/processed/cellline_programs/310_program_phenotype_associations.csv
data/processed/cellline_programs/310_program_discovery_metadata.json
````

The ICA artifacts define the primary candidate transcriptomic programs.

The NMF artifacts provide an independent representation for methodological contrast.

The matching artifact records quantitative cross-method structural support, while phenotype associations are evaluated only after the transcriptomic representations have been frozen.

## Methodological note

Programs identified in this notebook are candidate cell-line transcriptomic programs.

Association with the pharmacology-derived resistance-like phenotype does not establish drug-resistance mechanisms, causal regulation, clinical resistance, or therapeutic vulnerability.

Cross-method support strengthens computational robustness to representation choice but does not constitute independent biological validation.

The resulting candidate programs will require lineage-aware and covariate-aware robustness analysis in notebook 311 before entering cross-system comparison in Phase 4.

---

In [62]:
# =============================================================================
# Imports
# =============================================================================

from pancancer_epigenetics.utils.paths import (Paths, project_relative_path)

import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

from sklearn.decomposition import FastICA, NMF

In [3]:
# =============================================================================
# Input and output paths
# =============================================================================

TRANSCRIPTOME_PHENOTYPE_PATH = (
    Paths.pharmacology
    / "308_transcriptome_phenotype_dataset.parquet"
)

OUTPUT_DIR = Paths.cellline_programs

In [4]:
# =============================================================================
# Load frozen transcriptome–phenotype dataset
# =============================================================================

transcriptome_phenotype = pd.read_parquet(
    TRANSCRIPTOME_PHENOTYPE_PATH
)

print("Transcriptome–phenotype dataset:", transcriptome_phenotype.shape)

Transcriptome–phenotype dataset: (713, 19195)


In [5]:
# =============================================================================
# Separate transcriptome and phenotype
# =============================================================================

model_ids = transcriptome_phenotype["ModelID"].copy()

expression = transcriptome_phenotype.drop(
    columns=[
        "ModelID",
        "selected_phenotype",
    ]
)

selected_phenotype = transcriptome_phenotype[
    "selected_phenotype"
].copy()

print("Expression matrix:", expression.shape)

Expression matrix: (713, 19193)


In [7]:
# =============================================================================
# Remove zero-variance genes
# =============================================================================

expression = expression.loc[
    :,
    expression.var(axis=0).gt(0),
]

print("Expression matrix after zero-variance removal:", expression.shape)

Expression matrix after zero-variance removal: (713, 19183)


In [8]:
# =============================================================================
# Program-discovery parameters
# =============================================================================

N_DISCOVERY_GENES = 5000
N_COMPONENTS = 50

N_ICA_RUNS = 20
RANDOM_SEED = 20260812

In [9]:
# =============================================================================
# Select discovery gene universe
# =============================================================================

gene_variance = expression.var(axis=0)

discovery_genes = (
    gene_variance
    .nlargest(N_DISCOVERY_GENES)
    .index
)

discovery_expression = expression.loc[
    :,
    discovery_genes,
].copy()

print("Discovery expression matrix:", discovery_expression.shape)

Discovery expression matrix: (713, 5000)


In [10]:
# =============================================================================
# Prepare ICA discovery matrix
# =============================================================================

ica_expression = discovery_expression.to_numpy(
    dtype=np.float64,
    copy=True,
)

ica_seeds = (
    RANDOM_SEED
    + np.arange(N_ICA_RUNS)
)

In [11]:
# =============================================================================
# Run multistart ICA
# =============================================================================

ica_runs = []

for seed in ica_seeds:
    model = FastICA(
        n_components=N_COMPONENTS,
        whiten="unit-variance",
        random_state=int(seed),
        max_iter=2000,
    )

    scores = model.fit_transform(ica_expression)

    ica_runs.append(
        {
            "seed": int(seed),
            "scores": scores,
            "loadings": model.mixing_.copy(),
            "n_iter": model.n_iter_,
        }
    )

print("ICA runs completed:", len(ica_runs))
print(
    "Maximum iterations used:",
    max(run["n_iter"] for run in ica_runs),
)

ICA runs completed: 20
Maximum iterations used: 103


In [12]:
# =============================================================================
# Define ICA component matching
# =============================================================================

def match_ica_components(reference_loadings, target_loadings):
    reference_centered = (
        reference_loadings
        - reference_loadings.mean(axis=0, keepdims=True)
    )
    target_centered = (
        target_loadings
        - target_loadings.mean(axis=0, keepdims=True)
    )

    reference_unit = (
        reference_centered
        / np.linalg.norm(
            reference_centered,
            axis=0,
            keepdims=True,
        )
    )
    target_unit = (
        target_centered
        / np.linalg.norm(
            target_centered,
            axis=0,
            keepdims=True,
        )
    )

    loading_correlations = (
        reference_unit.T
        @ target_unit
    )

    reference_indices, target_indices = (
        linear_sum_assignment(
            -np.abs(loading_correlations)
        )
    )

    matched_correlations = loading_correlations[
        reference_indices,
        target_indices,
    ]

    return (
        target_indices,
        matched_correlations,
    )

In [13]:
# =============================================================================
# Select central ICA reference run
# =============================================================================

ica_run_stability = []

for reference_index, reference_run in enumerate(ica_runs):
    pairwise_stability = []

    for target_index, target_run in enumerate(ica_runs):
        if reference_index == target_index:
            continue

        _, matched_correlations = match_ica_components(
            reference_run["loadings"],
            target_run["loadings"],
        )

        pairwise_stability.append(
            np.median(np.abs(matched_correlations))
        )

    ica_run_stability.append(
        {
            "run_index": reference_index,
            "seed": reference_run["seed"],
            "median_pairwise_stability": np.median(
                pairwise_stability
            ),
        }
    )

ica_run_stability = pd.DataFrame(
    ica_run_stability
).sort_values(
    "median_pairwise_stability",
    ascending=False,
)

reference_run_index = int(
    ica_run_stability.iloc[0]["run_index"]
)

reference_ica_run = ica_runs[
    reference_run_index
]

ica_run_stability.head()

,run_index,seed,median_pairwise_stability
18,18,20260830,0.993054
7,7,20260819,0.992650
8,8,20260820,0.992401
19,19,20260831,0.992272
0,0,20260812,0.991986


In [15]:
# =============================================================================
# Quantify component-level ICA stability
# =============================================================================

component_stability = []

for target_index, target_run in enumerate(ica_runs):
    if target_index == reference_run_index:
        continue

    matched_indices, matched_correlations = match_ica_components(
        reference_ica_run["loadings"],
        target_run["loadings"],
    )

    for component_index, (
        matched_index,
        loading_correlation,
    ) in enumerate(
        zip(
            matched_indices,
            matched_correlations,
        )
    ):
        sign = np.sign(loading_correlation)

        score_correlation = np.corrcoef(
            reference_ica_run["scores"][:, component_index],
            target_run["scores"][:, matched_index] * sign,
        )[0, 1]

        component_stability.append(
            {
                "component_index": component_index,
                "loading_correlation": abs(loading_correlation),
                "score_correlation": score_correlation,
            }
        )

component_stability = pd.DataFrame(component_stability)

In [16]:
# =============================================================================
# Summarize component-level ICA stability
# =============================================================================

ica_component_stability = (
    component_stability
    .groupby("component_index")
    .agg(
        median_loading_correlation=(
            "loading_correlation",
            "median",
        ),
        minimum_loading_correlation=(
            "loading_correlation",
            "min",
        ),
        median_score_correlation=(
            "score_correlation",
            "median",
        ),
        minimum_score_correlation=(
            "score_correlation",
            "min",
        ),
    )
    .reset_index()
)

ica_component_stability.sort_values(
    [
        "median_loading_correlation",
        "median_score_correlation",
    ]
).head(10)

,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation
0,0,0.635381,0.261649,0.625547,0.223401
29,29,0.671491,0.323067,0.645514,0.217302
14,14,0.708586,0.505888,0.773943,0.520030
12,12,0.735925,0.219348,0.714653,0.456802
2,2,0.736396,0.471696,0.781630,0.471580
48,48,0.752521,0.334804,0.826855,0.433918
44,44,0.755739,0.562481,0.793832,0.587521
19,19,0.800160,0.573747,0.851715,0.334501
13,13,0.854009,0.781099,0.867742,0.772105
4,4,0.867388,0.263328,0.882903,0.409192


In [ ]:
# =============================================================================
# Inspect ICA stability distribution
# =============================================================================

ica_stability_distribution = (
    ica_component_stability[
        [
            "median_loading_correlation",
            "median_score_correlation",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

ica_stability_distribution

,median_loading_correlation,median_score_correlation
min,0.635381,0.625547
25%,0.907767,0.904567
50%,0.992928,0.989642
75%,0.999307,0.998373
max,0.999830,0.999544


In [20]:
# =============================================================================
# Define ICA component recoverability
# =============================================================================

ICA_STABILITY_THRESHOLD = 0.70

ica_component_stability["is_recoverable"] = (
    ica_component_stability[
        "median_loading_correlation"
    ].ge(ICA_STABILITY_THRESHOLD)
    & ica_component_stability[
        "median_score_correlation"
    ].ge(ICA_STABILITY_THRESHOLD)
)

ica_component_stability[
    "is_recoverable"
].value_counts()

is_recoverable
True     48
False     2
Name: count, dtype: int64

In [21]:
# =============================================================================
# Orient ICA components deterministically
# =============================================================================

ica_scores = reference_ica_run["scores"].copy()
ica_loadings = reference_ica_run["loadings"].copy()

for component_index in range(N_COMPONENTS):
    anchor_gene_index = np.argmax(
        np.abs(ica_loadings[:, component_index])
    )

    orientation = np.sign(
        ica_loadings[
            anchor_gene_index,
            component_index,
        ]
    )

    ica_scores[:, component_index] *= orientation
    ica_loadings[:, component_index] *= orientation

In [22]:
# =============================================================================
# Construct ICA program matrices
# =============================================================================

ica_program_ids = [
    f"ICA_PROGRAM_{index + 1:02d}"
    for index in range(N_COMPONENTS)
]

ica_program_scores = pd.DataFrame(
    ica_scores,
    index=model_ids,
    columns=ica_program_ids,
).reset_index(
    names="ModelID"
)

ica_program_loadings = pd.DataFrame(
    ica_loadings,
    index=discovery_expression.columns,
    columns=ica_program_ids,
).reset_index(
    names="gene"
)

print("ICA program scores  :", ica_program_scores.shape)
print("ICA program loadings:", ica_program_loadings.shape)

ICA program scores  : (713, 51)
ICA program loadings: (5000, 51)


In [23]:
# =============================================================================
# Construct ICA program metadata
# =============================================================================

ica_program_metadata = (
    ica_component_stability
    .assign(
        program_id=[
            f"ICA_PROGRAM_{index + 1:02d}"
            for index in ica_component_stability[
                "component_index"
            ]
        ]
    )
    [
        [
            "program_id",
            "component_index",
            "median_loading_correlation",
            "minimum_loading_correlation",
            "median_score_correlation",
            "minimum_score_correlation",
            "is_recoverable",
        ]
    ]
)

ica_program_metadata

,program_id,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation,is_recoverable
0,ICA_PROGRAM_01,0,0.635381,0.261649,0.625547,0.223401,False
1,ICA_PROGRAM_02,1,0.999823,0.999461,0.999544,0.999085,True
2,ICA_PROGRAM_03,2,0.736396,0.471696,0.781630,0.471580,True
3,ICA_PROGRAM_04,3,0.999677,0.999006,0.998404,0.996489,True
4,ICA_PROGRAM_05,4,0.867388,0.263328,0.882903,0.409192,True
5,ICA_PROGRAM_06,5,0.936306,0.845028,0.935106,0.822475,True
6,ICA_PROGRAM_07,6,0.982388,0.593796,0.976836,0.614546,True
7,ICA_PROGRAM_08,7,0.990809,0.970670,0.979823,0.947429,True
8,ICA_PROGRAM_09,8,0.999830,0.999350,0.998967,0.997130,True
9,ICA_PROGRAM_10,9,0.995252,0.987633,0.989461,0.975654,True


In [25]:
# =============================================================================
# Prepare NMF discovery matrix
# =============================================================================

N_NMF_RUNS = 20

nmf_expression = discovery_expression.to_numpy(
    dtype=np.float64,
    copy=True,
)

nmf_seeds = (
    RANDOM_SEED
    + np.arange(N_NMF_RUNS)
)

In [26]:
# =============================================================================
# Run multistart NMF
# =============================================================================

nmf_runs = []

for seed in nmf_seeds:
    model = NMF(
        n_components=N_COMPONENTS,
        init="nndsvdar",
        random_state=int(seed),
        max_iter=2000,
    )

    scores = model.fit_transform(nmf_expression)

    nmf_runs.append(
        {
            "seed": int(seed),
            "scores": scores,
            "loadings": model.components_.T.copy(),
            "reconstruction_error": model.reconstruction_err_,
            "n_iter": model.n_iter_,
        }
    )

print("NMF runs completed:", len(nmf_runs))
print(
    "Maximum iterations used:",
    max(run["n_iter"] for run in nmf_runs),
)

NMF runs completed: 20
Maximum iterations used: 1840


In [27]:
# =============================================================================
# Summarize NMF run diagnostics
# =============================================================================

nmf_run_diagnostics = pd.DataFrame(
    {
        "seed": [run["seed"] for run in nmf_runs],
        "n_iter": [run["n_iter"] for run in nmf_runs],
        "reconstruction_error": [
            run["reconstruction_error"]
            for run in nmf_runs
        ],
    }
)

nmf_run_diagnostics.describe()

,seed,n_iter,reconstruction_error
count,2.000000e+01,20.00000,20.000000
mean,2.026082e+07,1033.55000,1935.947394
std,5.916080e+00,332.13147,0.297245
min,2.026081e+07,482.00000,1935.497996
25%,2.026082e+07,805.00000,1935.741322
50%,2.026082e+07,950.50000,1935.801815
75%,2.026083e+07,1234.50000,1936.143224
max,2.026083e+07,1840.00000,1936.531686


In [29]:
# =============================================================================
# Define NMF component matching
# =============================================================================

def match_nmf_components(reference_loadings, target_loadings):
    reference_unit = (
        reference_loadings
        / np.linalg.norm(
            reference_loadings,
            axis=0,
            keepdims=True,
        )
    )

    target_unit = (
        target_loadings
        / np.linalg.norm(
            target_loadings,
            axis=0,
            keepdims=True,
        )
    )

    loading_similarity = (
        reference_unit.T
        @ target_unit
    )

    reference_indices, target_indices = (
        linear_sum_assignment(
            -loading_similarity
        )
    )

    matched_similarity = loading_similarity[
        reference_indices,
        target_indices,
    ]

    return (
        target_indices,
        matched_similarity,
    )

In [30]:
# =============================================================================
# Select central NMF reference run
# =============================================================================

nmf_run_stability = []

for reference_index, reference_run in enumerate(nmf_runs):
    pairwise_stability = []

    for target_index, target_run in enumerate(nmf_runs):
        if reference_index == target_index:
            continue

        _, matched_similarity = match_nmf_components(
            reference_run["loadings"],
            target_run["loadings"],
        )

        pairwise_stability.append(
            np.median(matched_similarity)
        )

    nmf_run_stability.append(
        {
            "run_index": reference_index,
            "seed": reference_run["seed"],
            "median_pairwise_stability": np.median(
                pairwise_stability
            ),
        }
    )

nmf_run_stability = pd.DataFrame(
    nmf_run_stability
).sort_values(
    "median_pairwise_stability",
    ascending=False,
)

reference_nmf_run_index = int(
    nmf_run_stability.iloc[0]["run_index"]
)

reference_nmf_run = nmf_runs[
    reference_nmf_run_index
]

nmf_run_stability.head()

,run_index,seed,median_pairwise_stability
6,6,20260818,0.998392
18,18,20260830,0.998360
13,13,20260825,0.998359
7,7,20260819,0.998359
11,11,20260823,0.998146


In [32]:
# =============================================================================
# Quantify component-level NMF stability
# =============================================================================

nmf_component_stability_records = []

for target_index, target_run in enumerate(nmf_runs):
    if target_index == reference_nmf_run_index:
        continue

    matched_indices, matched_similarity = match_nmf_components(
        reference_nmf_run["loadings"],
        target_run["loadings"],
    )

    for component_index, (
        matched_index,
        loading_similarity,
    ) in enumerate(
        zip(
            matched_indices,
            matched_similarity,
        )
    ):
        score_correlation = np.corrcoef(
            reference_nmf_run["scores"][:, component_index],
            target_run["scores"][:, matched_index],
        )[0, 1]

        nmf_component_stability_records.append(
            {
                "component_index": component_index,
                "loading_similarity": loading_similarity,
                "score_correlation": score_correlation,
            }
        )

nmf_component_stability_records = pd.DataFrame(
    nmf_component_stability_records
)

In [33]:
# =============================================================================
# Summarize component-level NMF stability
# =============================================================================

nmf_component_stability = (
    nmf_component_stability_records
    .groupby("component_index")
    .agg(
        median_loading_similarity=(
            "loading_similarity",
            "median",
        ),
        minimum_loading_similarity=(
            "loading_similarity",
            "min",
        ),
        median_score_correlation=(
            "score_correlation",
            "median",
        ),
        minimum_score_correlation=(
            "score_correlation",
            "min",
        ),
    )
    .reset_index()
)

nmf_component_stability.sort_values(
    [
        "median_loading_similarity",
        "median_score_correlation",
    ]
).head(10)

,component_index,median_loading_similarity,minimum_loading_similarity,median_score_correlation,minimum_score_correlation
35,35,0.834749,0.587494,0.616776,-0.006336
29,29,0.957591,0.648231,0.882325,0.116107
0,0,0.975246,0.517282,0.935058,-0.129004
28,28,0.989458,0.787484,0.985427,0.565242
8,8,0.991168,0.791121,0.976849,0.610794
10,10,0.992249,0.982965,0.979064,0.961896
47,47,0.992700,0.959610,0.987234,0.894078
9,9,0.992720,0.770715,0.972551,0.404529
30,30,0.993002,0.974306,0.987503,0.973856
13,13,0.993616,0.986954,0.993760,0.989352


In [35]:
# =============================================================================
# Inspect NMF stability distribution
# =============================================================================

nmf_stability_distribution = (
    nmf_component_stability[
        [
            "median_loading_similarity",
            "median_score_correlation",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

nmf_stability_distribution

,median_loading_similarity,median_score_correlation
min,0.834749,0.616776
25%,0.994814,0.988234
50%,0.998062,0.997188
75%,0.999637,0.999391
max,0.999946,0.999956


In [37]:
# =============================================================================
# Define NMF component recoverability
# =============================================================================

NMF_STABILITY_THRESHOLD = 0.70

nmf_component_stability["is_recoverable"] = (
    nmf_component_stability[
        "median_loading_similarity"
    ].ge(NMF_STABILITY_THRESHOLD)
    & nmf_component_stability[
        "median_score_correlation"
    ].ge(NMF_STABILITY_THRESHOLD)
)

nmf_component_stability[
    "is_recoverable"
].value_counts()

is_recoverable
True     49
False     1
Name: count, dtype: int64

In [38]:
# =============================================================================
# Construct NMF program matrices
# =============================================================================

nmf_program_ids = [
    f"NMF_PROGRAM_{index + 1:02d}"
    for index in range(N_COMPONENTS)
]

nmf_program_scores = pd.DataFrame(
    reference_nmf_run["scores"],
    index=model_ids,
    columns=nmf_program_ids,
).reset_index(
    names="ModelID"
)

nmf_program_loadings = pd.DataFrame(
    reference_nmf_run["loadings"],
    index=discovery_expression.columns,
    columns=nmf_program_ids,
).reset_index(
    names="gene"
)

print("NMF program scores  :", nmf_program_scores.shape)
print("NMF program loadings:", nmf_program_loadings.shape)

NMF program scores  : (713, 51)
NMF program loadings: (5000, 51)


In [39]:
# =============================================================================
# Construct NMF program metadata
# =============================================================================

nmf_program_metadata = (
    nmf_component_stability
    .assign(
        program_id=[
            f"NMF_PROGRAM_{index + 1:02d}"
            for index in nmf_component_stability[
                "component_index"
            ]
        ]
    )
    [
        [
            "program_id",
            "component_index",
            "median_loading_similarity",
            "minimum_loading_similarity",
            "median_score_correlation",
            "minimum_score_correlation",
            "is_recoverable",
        ]
    ]
)

nmf_program_metadata

,program_id,component_index,median_loading_similarity,minimum_loading_similarity,median_score_correlation,minimum_score_correlation,is_recoverable
0,NMF_PROGRAM_01,0,0.975246,0.517282,0.935058,-0.129004,True
1,NMF_PROGRAM_02,1,0.995024,0.790892,0.978484,0.452644,True
2,NMF_PROGRAM_03,2,0.996651,0.986379,0.990506,0.966537,True
3,NMF_PROGRAM_04,3,0.999523,0.998766,0.998203,0.995521,True
4,NMF_PROGRAM_05,4,0.998816,0.981459,0.999081,0.911172,True
5,NMF_PROGRAM_06,5,0.999881,0.999613,0.999756,0.999064,True
6,NMF_PROGRAM_07,6,0.998328,0.932076,0.997449,0.829033,True
7,NMF_PROGRAM_08,7,0.999915,0.999676,0.999956,0.999880,True
8,NMF_PROGRAM_09,8,0.991168,0.791121,0.976849,0.610794,True
9,NMF_PROGRAM_10,9,0.992720,0.770715,0.972551,0.404529,True


In [41]:
# =============================================================================
# Define recoverable programs for cross-method comparison
# =============================================================================

recoverable_ica_programs = (
    ica_program_metadata
    .loc[
        ica_program_metadata["is_recoverable"],
        "program_id",
    ]
    .tolist()
)

recoverable_nmf_programs = (
    nmf_program_metadata
    .loc[
        nmf_program_metadata["is_recoverable"],
        "program_id",
    ]
    .tolist()
)

In [42]:
# =============================================================================
# Compute pairwise ICA–NMF structural concordance
# =============================================================================

cross_method_records = []

for ica_program in recoverable_ica_programs:
    ica_loading = ica_program_loadings[
        ica_program
    ].to_numpy()

    ica_positive = np.clip(
        ica_loading,
        a_min=0,
        a_max=None,
    )
    ica_negative = np.clip(
        -ica_loading,
        a_min=0,
        a_max=None,
    )

    ica_score = ica_program_scores[
        ica_program
    ].to_numpy()

    for nmf_program in recoverable_nmf_programs:
        nmf_loading = nmf_program_loadings[
            nmf_program
        ].to_numpy()

        positive_similarity = (
            ica_positive @ nmf_loading
        ) / (
            np.linalg.norm(ica_positive)
            * np.linalg.norm(nmf_loading)
        )

        negative_similarity = (
            ica_negative @ nmf_loading
        ) / (
            np.linalg.norm(ica_negative)
            * np.linalg.norm(nmf_loading)
        )

        matched_pole = (
            "positive"
            if positive_similarity >= negative_similarity
            else "negative"
        )

        loading_similarity = max(
            positive_similarity,
            negative_similarity,
        )

        score_correlation, _ = spearmanr(
            ica_score,
            nmf_program_scores[
                nmf_program
            ].to_numpy(),
        )

        aligned_score_correlation = (
            score_correlation
            if matched_pole == "positive"
            else -score_correlation
        )

        cross_method_records.append(
            {
                "ica_program": ica_program,
                "nmf_program": nmf_program,
                "ica_pole": matched_pole,
                "loading_similarity": loading_similarity,
                "score_spearman": score_correlation,
                "aligned_score_spearman": (
                    aligned_score_correlation
                ),
            }
        )

cross_method_concordance = pd.DataFrame(
    cross_method_records
)

In [43]:
# =============================================================================
# Summarize best ICA–NMF matches across structural views
# =============================================================================

best_loading_matches = (
    cross_method_concordance
    .sort_values(
        "loading_similarity",
        ascending=False,
    )
    .groupby("ica_program", as_index=False)
    .first()
    .rename(
        columns={
            "nmf_program": "best_loading_nmf",
            "ica_pole": "best_loading_pole",
            "loading_similarity": "best_loading_similarity",
            "aligned_score_spearman": "score_at_best_loading",
        }
    )
)

best_score_matches = (
    cross_method_concordance
    .sort_values(
        "aligned_score_spearman",
        ascending=False,
    )
    .groupby("ica_program", as_index=False)
    .first()
    .rename(
        columns={
            "nmf_program": "best_score_nmf",
            "loading_similarity": "loading_at_best_score",
            "aligned_score_spearman": "best_aligned_score_spearman",
        }
    )
)

cross_method_match_summary = (
    best_loading_matches[
        [
            "ica_program",
            "best_loading_nmf",
            "best_loading_pole",
            "best_loading_similarity",
            "score_at_best_loading",
        ]
    ]
    .merge(
        best_score_matches[
            [
                "ica_program",
                "best_score_nmf",
                "loading_at_best_score",
                "best_aligned_score_spearman",
            ]
        ],
        on="ica_program",
    )
)

cross_method_match_summary["same_nmf_across_views"] = (
    cross_method_match_summary["best_loading_nmf"]
    == cross_method_match_summary["best_score_nmf"]
)

cross_method_match_summary.sort_values(
    "best_loading_similarity"
).head(10)

,ica_program,best_loading_nmf,best_loading_pole,best_loading_similarity,score_at_best_loading,best_score_nmf,loading_at_best_score,best_aligned_score_spearman,same_nmf_across_views
13,ICA_PROGRAM_15,NMF_PROGRAM_31,positive,0.596093,0.421028,NMF_PROGRAM_31,0.596093,0.421028,True
37,ICA_PROGRAM_40,NMF_PROGRAM_30,negative,0.598063,0.126550,NMF_PROGRAM_47,0.559321,0.234538,False
18,ICA_PROGRAM_20,NMF_PROGRAM_14,positive,0.603958,0.366833,NMF_PROGRAM_14,0.603958,0.366833,True
24,ICA_PROGRAM_26,NMF_PROGRAM_48,positive,0.609160,0.160983,NMF_PROGRAM_10,0.575300,0.398990,False
3,ICA_PROGRAM_05,NMF_PROGRAM_11,negative,0.620992,0.373827,NMF_PROGRAM_11,0.620992,0.373827,True
42,ICA_PROGRAM_45,NMF_PROGRAM_09,positive,0.647534,0.457594,NMF_PROGRAM_09,0.647534,0.457594,True
31,ICA_PROGRAM_34,NMF_PROGRAM_48,positive,0.676204,0.167250,NMF_PROGRAM_15,0.512340,0.662110,False
33,ICA_PROGRAM_36,NMF_PROGRAM_03,positive,0.681654,0.258445,NMF_PROGRAM_14,0.491416,0.314685,False
1,ICA_PROGRAM_03,NMF_PROGRAM_41,positive,0.700767,0.530974,NMF_PROGRAM_41,0.700767,0.530974,True
5,ICA_PROGRAM_07,NMF_PROGRAM_03,positive,0.702214,0.163278,NMF_PROGRAM_02,0.418570,0.404605,False


In [45]:
# =============================================================================
# Summarize ICA–NMF cross-method concordance
# =============================================================================

cross_method_overview = pd.Series(
    {
        "recoverable_ica_programs": len(
            recoverable_ica_programs
        ),
        "same_nmf_across_views": (
            cross_method_match_summary[
                "same_nmf_across_views"
            ].sum()
        ),
        "median_best_loading_similarity": (
            cross_method_match_summary[
                "best_loading_similarity"
            ].median()
        ),
        "minimum_best_loading_similarity": (
            cross_method_match_summary[
                "best_loading_similarity"
            ].min()
        ),
        "median_score_at_best_loading": (
            cross_method_match_summary[
                "score_at_best_loading"
            ].median()
        ),
        "minimum_score_at_best_loading": (
            cross_method_match_summary[
                "score_at_best_loading"
            ].min()
        ),
    }
)

cross_method_overview

recoverable_ica_programs           48.000000
same_nmf_across_views              38.000000
median_best_loading_similarity      0.755456
minimum_best_loading_similarity     0.596093
median_score_at_best_loading        0.466087
minimum_score_at_best_loading       0.118844
dtype: float64

In [46]:
# =============================================================================
# Quantify ICA–NMF match uniqueness
# =============================================================================

match_uniqueness_records = []

for ica_program, program_matches in (
    cross_method_concordance
    .groupby("ica_program")
):
    loading_ranked = program_matches.sort_values(
        "loading_similarity",
        ascending=False,
    )

    score_ranked = program_matches.sort_values(
        "aligned_score_spearman",
        ascending=False,
    )

    match_uniqueness_records.append(
        {
            "ica_program": ica_program,
            "loading_match_margin": (
                loading_ranked.iloc[0]["loading_similarity"]
                - loading_ranked.iloc[1]["loading_similarity"]
            ),
            "score_match_margin": (
                score_ranked.iloc[0]["aligned_score_spearman"]
                - score_ranked.iloc[1]["aligned_score_spearman"]
            ),
        }
    )

match_uniqueness = pd.DataFrame(
    match_uniqueness_records
)

cross_method_match_summary = (
    cross_method_match_summary
    .merge(
        match_uniqueness,
        on="ica_program",
        how="left",
    )
)

cross_method_match_summary[
    [
        "loading_match_margin",
        "score_match_margin",
    ]
].describe()

,loading_match_margin,score_match_margin
count,48.000000,48.000000
mean,0.111455,0.243819
std,0.078065,0.114771
min,0.000705,0.028748
25%,0.038647,0.139285
50%,0.110076,0.261784
75%,0.153739,0.340244
max,0.313812,0.466050


In [48]:
# =============================================================================
# Define convergent ICA–NMF structural matches
# =============================================================================

cross_method_match_summary[
    "cross_method_convergent"
] = (
    cross_method_match_summary[
        "same_nmf_across_views"
    ]
)

cross_method_match_summary[
    [
        "ica_program",
        "best_loading_nmf",
        "best_loading_pole",
        "best_loading_similarity",
        "score_at_best_loading",
        "loading_match_margin",
        "score_match_margin",
        "cross_method_convergent",
    ]
].sort_values(
    [
        "cross_method_convergent",
        "best_loading_similarity",
    ],
    ascending=[
        False,
        False,
    ],
)

,ica_program,best_loading_nmf,best_loading_pole,best_loading_similarity,score_at_best_loading,loading_match_margin,score_match_margin,cross_method_convergent
8,ICA_PROGRAM_10,NMF_PROGRAM_35,positive,0.864225,0.470203,0.249001,0.304630,True
36,ICA_PROGRAM_39,NMF_PROGRAM_07,positive,0.840810,0.461971,0.166101,0.226142,True
40,ICA_PROGRAM_43,NMF_PROGRAM_45,positive,0.837902,0.583858,0.116597,0.419468,True
29,ICA_PROGRAM_32,NMF_PROGRAM_37,positive,0.835198,0.551404,0.313812,0.395881,True
32,ICA_PROGRAM_35,NMF_PROGRAM_44,positive,0.830133,0.559697,0.211270,0.339849,True
21,ICA_PROGRAM_23,NMF_PROGRAM_40,positive,0.828100,0.538483,0.183289,0.368110,True
2,ICA_PROGRAM_04,NMF_PROGRAM_04,positive,0.819850,0.531295,0.145639,0.355793,True
14,ICA_PROGRAM_16,NMF_PROGRAM_34,positive,0.815397,0.541965,0.222995,0.342458,True
45,ICA_PROGRAM_48,NMF_PROGRAM_28,positive,0.814371,0.416458,0.024141,0.162990,True
28,ICA_PROGRAM_31,NMF_PROGRAM_21,positive,0.800292,0.520966,0.187109,0.260003,True


In [50]:
# =============================================================================
# Assemble ICA program discovery summary
# =============================================================================

ica_program_discovery_summary = (
    ica_program_metadata
    .merge(
        cross_method_match_summary,
        left_on="program_id",
        right_on="ica_program",
        how="left",
    )
    .drop(columns="ica_program")
)

ica_program_discovery_summary[
    "program_status"
] = np.where(
    ~ica_program_discovery_summary["is_recoverable"],
    "not_recoverable",
    np.where(
        ica_program_discovery_summary[
            "cross_method_convergent"
        ].fillna(False),
        "candidate_with_cross_method_support",
        "candidate_ica_specific",
    ),
)

ica_program_discovery_summary[
    "program_status"
].value_counts()

program_status
candidate_with_cross_method_support    38
candidate_ica_specific                 10
not_recoverable                         2
Name: count, dtype: int64

In [51]:
# =============================================================================
# Associate recoverable ICA programs with selected phenotype
# =============================================================================

phenotype_associations = []

for program_id in recoverable_ica_programs:
    rho, p_value = spearmanr(
        ica_program_scores[program_id],
        selected_phenotype,
    )

    phenotype_associations.append(
        {
            "program_id": program_id,
            "spearman_rho": rho,
            "p_value": p_value,
        }
    )

phenotype_associations = pd.DataFrame(
    phenotype_associations
)

phenotype_associations[
    "absolute_spearman_rho"
] = phenotype_associations[
    "spearman_rho"
].abs()

phenotype_associations.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,spearman_rho,p_value,absolute_spearman_rho
43,ICA_PROGRAM_46,0.211964,1.096989e-08,0.211964
7,ICA_PROGRAM_09,-0.192371,2.265605e-07,0.192371
30,ICA_PROGRAM_33,-0.145618,9.527935e-05,0.145618
27,ICA_PROGRAM_29,-0.132617,3.843132e-04,0.132617
39,ICA_PROGRAM_42,-0.132021,4.084942e-04,0.132021
4,ICA_PROGRAM_06,0.129352,5.352398e-04,0.129352
11,ICA_PROGRAM_13,-0.112589,2.606871e-03,0.112589
18,ICA_PROGRAM_20,0.103747,5.556169e-03,0.103747
5,ICA_PROGRAM_07,0.100847,7.039402e-03,0.100847
16,ICA_PROGRAM_18,-0.096073,1.026482e-02,0.096073


In [53]:
# =============================================================================
# Adjust phenotype associations for multiple testing
# =============================================================================

p_values = phenotype_associations[
    "p_value"
].to_numpy()

order = np.argsort(p_values)
ranked_p_values = p_values[order]

adjusted_p_values = (
    ranked_p_values
    * len(ranked_p_values)
    / np.arange(1, len(ranked_p_values) + 1)
)

adjusted_p_values = np.minimum.accumulate(
    adjusted_p_values[::-1]
)[::-1]

q_values = np.empty_like(adjusted_p_values)
q_values[order] = np.clip(
    adjusted_p_values,
    0,
    1,
)

phenotype_associations["q_value"] = q_values

phenotype_associations.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,spearman_rho,p_value,absolute_spearman_rho,q_value
43,ICA_PROGRAM_46,0.211964,1.096989e-08,0.211964,5.265548e-07
7,ICA_PROGRAM_09,-0.192371,2.265605e-07,0.192371,5.437453e-06
30,ICA_PROGRAM_33,-0.145618,9.527935e-05,0.145618,1.524470e-03
27,ICA_PROGRAM_29,-0.132617,3.843132e-04,0.132617,3.921545e-03
39,ICA_PROGRAM_42,-0.132021,4.084942e-04,0.132021,3.921545e-03
4,ICA_PROGRAM_06,0.129352,5.352398e-04,0.129352,4.281918e-03
11,ICA_PROGRAM_13,-0.112589,2.606871e-03,0.112589,1.787569e-02
18,ICA_PROGRAM_20,0.103747,5.556169e-03,0.103747,3.333702e-02
5,ICA_PROGRAM_07,0.100847,7.039402e-03,0.100847,3.754348e-02
16,ICA_PROGRAM_18,-0.096073,1.026482e-02,0.096073,4.927114e-02


In [55]:
# =============================================================================
# Integrate phenotype associations with program discovery summary
# =============================================================================

program_discovery_summary = (
    ica_program_discovery_summary
    .merge(
        phenotype_associations,
        on="program_id",
        how="left",
    )
)

program_discovery_summary.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation,is_recoverable,best_loading_nmf,best_loading_pole,best_loading_similarity,...,best_aligned_score_spearman,same_nmf_across_views,loading_match_margin,score_match_margin,cross_method_convergent,program_status,spearman_rho,p_value,absolute_spearman_rho,q_value
45,ICA_PROGRAM_46,45,0.915088,0.635852,0.901495,0.520448,True,NMF_PROGRAM_48,positive,0.718667,...,0.423155,False,0.027549,0.032430,False,candidate_ica_specific,0.211964,1.096989e-08,0.211964,5.265548e-07
8,ICA_PROGRAM_09,8,0.999830,0.999350,0.998967,0.997130,True,NMF_PROGRAM_50,positive,0.743526,...,0.472635,True,0.013132,0.115271,True,candidate_with_cross_method_support,-0.192371,2.265605e-07,0.192371,5.437453e-06
32,ICA_PROGRAM_33,32,0.999405,0.997669,0.998403,0.996440,True,NMF_PROGRAM_43,positive,0.731088,...,0.523852,True,0.052227,0.343562,True,candidate_with_cross_method_support,-0.145618,9.527935e-05,0.145618,1.524470e-03
28,ICA_PROGRAM_29,28,0.992061,0.846806,0.975545,0.658390,True,NMF_PROGRAM_12,positive,0.754323,...,0.307432,True,0.093771,0.028748,True,candidate_with_cross_method_support,-0.132617,3.843132e-04,0.132617,3.921545e-03
41,ICA_PROGRAM_42,41,0.999709,0.999251,0.999204,0.998297,True,NMF_PROGRAM_24,positive,0.736297,...,0.543028,True,0.034974,0.377195,True,candidate_with_cross_method_support,-0.132021,4.084942e-04,0.132021,3.921545e-03
5,ICA_PROGRAM_06,5,0.936306,0.845028,0.935106,0.822475,True,NMF_PROGRAM_33,positive,0.770524,...,0.441160,True,0.152569,0.134298,True,candidate_with_cross_method_support,0.129352,5.352398e-04,0.129352,4.281918e-03
12,ICA_PROGRAM_13,12,0.735925,0.219348,0.714653,0.456802,True,NMF_PROGRAM_07,negative,0.762666,...,0.401909,False,0.049513,0.169213,False,candidate_ica_specific,-0.112589,2.606871e-03,0.112589,1.787569e-02
19,ICA_PROGRAM_20,19,0.800160,0.573747,0.851715,0.334501,True,NMF_PROGRAM_14,positive,0.603958,...,0.366833,True,0.020436,0.120529,True,candidate_with_cross_method_support,0.103747,5.556169e-03,0.103747,3.333702e-02
6,ICA_PROGRAM_07,6,0.982388,0.593796,0.976836,0.614546,True,NMF_PROGRAM_03,positive,0.702214,...,0.404605,False,0.028897,0.106724,False,candidate_ica_specific,0.100847,7.039402e-03,0.100847,3.754348e-02
17,ICA_PROGRAM_18,17,0.999544,0.998371,0.998731,0.997422,True,NMF_PROGRAM_06,positive,0.767544,...,0.480255,True,0.152405,0.228151,True,candidate_with_cross_method_support,-0.096073,1.026482e-02,0.096073,4.927114e-02


In [57]:
# =============================================================================
# Identify phenotype-associated candidate programs
# =============================================================================

PHENOTYPE_FDR_THRESHOLD = 0.05

program_discovery_summary[
    "passes_phenotype_fdr"
] = (
    program_discovery_summary["q_value"]
    .lt(PHENOTYPE_FDR_THRESHOLD)
)

program_discovery_summary.loc[
    program_discovery_summary["passes_phenotype_fdr"],
    [
        "program_id",
        "program_status",
        "spearman_rho",
        "q_value",
    ],
].sort_values(
    "spearman_rho",
    key=abs,
    ascending=False,
)

,program_id,program_status,spearman_rho,q_value
45,ICA_PROGRAM_46,candidate_ica_specific,0.211964,5.265548e-07
8,ICA_PROGRAM_09,candidate_with_cross_method_support,-0.192371,5.437453e-06
32,ICA_PROGRAM_33,candidate_with_cross_method_support,-0.145618,1.524470e-03
28,ICA_PROGRAM_29,candidate_with_cross_method_support,-0.132617,3.921545e-03
41,ICA_PROGRAM_42,candidate_with_cross_method_support,-0.132021,3.921545e-03
5,ICA_PROGRAM_06,candidate_with_cross_method_support,0.129352,4.281918e-03
12,ICA_PROGRAM_13,candidate_ica_specific,-0.112589,1.787569e-02
19,ICA_PROGRAM_20,candidate_with_cross_method_support,0.103747,3.333702e-02
6,ICA_PROGRAM_07,candidate_ica_specific,0.100847,3.754348e-02
17,ICA_PROGRAM_18,candidate_with_cross_method_support,-0.096073,4.927114e-02


In [59]:
# =============================================================================
# Assemble program-discovery metadata
# =============================================================================

program_discovery_metadata = {
    "n_models": len(model_ids),
    "n_input_genes": expression.shape[1],
    "n_discovery_genes": N_DISCOVERY_GENES,
    "n_components": N_COMPONENTS,
    "random_seed": RANDOM_SEED,
    "ica": {
        "n_runs": N_ICA_RUNS,
        "reference_seed": reference_ica_run["seed"],
        "stability_threshold": ICA_STABILITY_THRESHOLD,
        "n_recoverable": int(
            ica_program_metadata["is_recoverable"].sum()
        ),
    },
    "nmf": {
        "n_runs": N_NMF_RUNS,
        "reference_seed": reference_nmf_run["seed"],
        "stability_threshold": NMF_STABILITY_THRESHOLD,
        "n_recoverable": int(
            nmf_program_metadata["is_recoverable"].sum()
        ),
    },
    "cross_method": {
        "n_convergent_ica_programs": int(
            program_discovery_summary[
                "cross_method_convergent"
            ].fillna(False).sum()
        ),
    },
    "phenotype_association": {
        "phenotype": "selected_phenotype",
        "method": "spearman",
        "fdr_method": "BH",
        "fdr_threshold": PHENOTYPE_FDR_THRESHOLD,
        "n_fdr_candidates": int(
            program_discovery_summary[
                "passes_phenotype_fdr"
            ].sum()
        ),
    },
}

In [60]:
# =============================================================================
# Program-discovery output paths
# =============================================================================

ICA_SCORES_OUTPUT_PATH = (
    OUTPUT_DIR / "310_ica_program_scores.parquet"
)

ICA_LOADINGS_OUTPUT_PATH = (
    OUTPUT_DIR / "310_ica_program_loadings.parquet"
)

NMF_SCORES_OUTPUT_PATH = (
    OUTPUT_DIR / "310_nmf_program_scores.parquet"
)

NMF_LOADINGS_OUTPUT_PATH = (
    OUTPUT_DIR / "310_nmf_program_loadings.parquet"
)

CROSS_METHOD_OUTPUT_PATH = (
    OUTPUT_DIR / "310_cross_method_program_matching.csv"
)

PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH = (
    OUTPUT_DIR / "310_program_phenotype_associations.csv"
)

METADATA_OUTPUT_PATH = (
    OUTPUT_DIR / "310_program_discovery_metadata.json"
)

In [63]:
# =============================================================================
# Save program-discovery artifacts
# =============================================================================

ica_program_scores.to_parquet(
    ICA_SCORES_OUTPUT_PATH,
    index=False,
)

ica_program_loadings.to_parquet(
    ICA_LOADINGS_OUTPUT_PATH,
    index=False,
)

nmf_program_scores.to_parquet(
    NMF_SCORES_OUTPUT_PATH,
    index=False,
)

nmf_program_loadings.to_parquet(
    NMF_LOADINGS_OUTPUT_PATH,
    index=False,
)

cross_method_match_summary.to_csv(
    CROSS_METHOD_OUTPUT_PATH,
    index=False,
)

program_discovery_summary.to_csv(
    PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH,
    index=False,
)

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        program_discovery_metadata,
        file,
        indent=2,
    )

print("Program-discovery artifacts written.")
print(f"ICA scores            : {project_relative_path(ICA_SCORES_OUTPUT_PATH)}")
print(f"ICA loadings          : {project_relative_path(ICA_LOADINGS_OUTPUT_PATH)}")
print(f"NMF scores            : {project_relative_path(NMF_SCORES_OUTPUT_PATH)}")
print(f"NMF loadings          : {project_relative_path(NMF_LOADINGS_OUTPUT_PATH)}")
print(f"Cross-method matching : {project_relative_path(CROSS_METHOD_OUTPUT_PATH)}")
print(f"Phenotype associations: {project_relative_path(PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH)}")
print(f"Metadata              : {project_relative_path(METADATA_OUTPUT_PATH)}")

Program-discovery artifacts written.
ICA scores            : data/processed/cellline_programs/310_ica_program_scores.parquet
ICA loadings          : data/processed/cellline_programs/310_ica_program_loadings.parquet
NMF scores            : data/processed/cellline_programs/310_nmf_program_scores.parquet
NMF loadings          : data/processed/cellline_programs/310_nmf_program_loadings.parquet
Cross-method matching : data/processed/cellline_programs/310_cross_method_program_matching.csv
Phenotype associations: data/processed/cellline_programs/310_program_phenotype_associations.csv
Metadata              : data/processed/cellline_programs/310_program_discovery_metadata.json


## Summary

Cell-line transcriptomic program discovery was completed on the frozen 713-model cohort using a phenotype-independent decomposition strategy.

ICA was used as the primary discovery framework with 50 components and 20 deterministic initializations. Global multistart stability was high, although component-level recoverability identified two ICA components below the prespecified multiview stability threshold. This resulted in 48 recoverable ICA candidate programs.

NMF was applied independently to the same 5,000-gene discovery universe as a prespecified methodological contrast. Forty-nine of 50 NMF factors were recoverable across multiple initializations.

Cross-method comparison was performed using both model-level score concordance and gene-level structure rather than component indices. Among the 48 recoverable ICA programs:

- 38 showed convergent ICA–NMF structural support;
- 10 remained ICA-specific;
- absence of NMF convergence was not used as an exclusion criterion.

The resistance-like phenotype was introduced only after transcriptomic representations and cross-method relationships had been frozen.

Ten recoverable ICA programs showed discovery-level association with the selected median raw LN_IC50 phenotype at BH-FDR < 0.05. Effect sizes were modest, with the strongest association reaching |Spearman ρ| ≈ 0.21.

The strongest phenotype-associated program, `ICA_PROGRAM_46`, was ICA-specific, demonstrating why NMF support should be interpreted as complementary structural evidence rather than as a mandatory validation filter.

These associations define candidate cell-line transcriptomic programs for downstream robustness analysis. They should not yet be interpreted as robust resistance programs, causal mechanisms, therapeutic vulnerabilities, or cross-cancer recurrent programs.

Notebook 311 will evaluate lineage dependence, phenotype-definition sensitivity, drug-response coverage, additional cell-line covariates, and resampling-based robustness before any program is considered for cross-system comparison in Phase 4.